<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Human-in-the-Loop
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Human-in-the-Loop ist die konkrete Umsetzung von **Pruefen** im Meeting- & Research-Briefing-Agent — eine unsichere oder folgenreiche Antwort geht nicht automatisch raus, sondern wird vor der Ausgabe gestoppt und freigegeben. Freigabe ist damit kein Zusatzfeature, sondern ein Produktmerkmal: Genau das unterscheidet den Assistant von einem GenAI-Chatbot, der einfach antwortet. Der `meeting-briefing`-Skill liefert dabei das fachliche Ausgabeformat; HITL entscheidet, wann ein Briefing vor Versand geprüft oder gestoppt wird.


In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M17-Human-in-the-Loop"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()
# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.9.3.81
Hostname: 81.3.9.34.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


# 1 | Übersicht
---

***Checkpointing & Sessions*** führte Checkpointing ein – der State bleibt über Aufrufe hinweg erhalten.  
**Dieses Modul** ergänzt das entscheidende Kontrollmuster: **der Mensch wird in den Ablauf eingebunden.**

Ohne HITL läuft ein Agent vollautomatisch – inklusive kritischer Aktionen.  
Mit HITL pausiert der Graph an definierten Punkten und wartet auf menschliche Entscheidung.

**Typische HITL-Szenarien:**

| Szenario | Kritische Aktion | HITL-Schritt |
|----------|-----------------|---------------|
| **Content-Freigabe** | Text veröffentlichen | Mensch liest + genehmigt Entwurf |
| **Tool-Genehmigung** | E-Mail senden, Datei löschen | Mensch bestätigt Tool-Aufruf |
| **Daten-Review** | Datenbankschreibzugriff | Mensch prüft Vorschau |
| **Eskalation** | Unklare Anfragen | Mensch übernimmt manuell |

**Zwei Wege, HITL umzusetzen:**

| Methode | Beschreibung | Einsatz |
|---------|-------------|--------|
| **`interrupt()`** in Node | Node fragt aktiv nach | Approval, Edit, Eskalation |
| **`interrupt_before`** | Graph pausiert deklarativ vor Node | Tool-Genehmigung |

> **Voraussetzung:** Ein Checkpointer (z.B. `InMemorySaver`) ist zwingend erforderlich –  
> ohne gespeicherten State kann der Graph nicht fortgesetzt werden.

In [2]:
#@markdown   <p><font size="4" color='green'>  HITL Workflow</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    START([START]) --> A["🤖 Agent Node"]
    A --> INT["⏸️ interrupt()\nGraph pausiert"]
    INT -->|"Checkpoint gespeichert"| H["👤 Mensch\nprüft & entscheidet"]
    H -->|"Command(resume='ja')"| R["▶️ Graph\nfortsetzt"]
    H -->|"Command(resume='nein')"| N["❌ Abbruch"]
    R --> END([END])
    N --> END

    style INT fill:#FF9800,color:#fff
    style H   fill:#2196F3,color:#fff
    style R   fill:#4CAF50,color:#fff
    style N   fill:#F44336,color:#fff
'''

mermaid(diagram, width=650)

# 2 | HITL Konzept
---

**Wie funktioniert `interrupt()`?**

```python
from langgraph.types import interrupt, Command

def genehmigung_node(state):
    # Pausiert den Graphen und gibt den Wert an den Aufrufer
    antwort = interrupt({"frage": "Genehmigen?", "daten": state["entwurf"]})
    return {"genehmigt": antwort == "ja"}
```

**Ablauf Schritt für Schritt:**

1. `graph.invoke(input, config=cfg)` – startet Ausführung
2. Node ruft `interrupt(wert)` auf – Graph pausiert, Checkpoint wird gespeichert
3. `invoke()` gibt Ergebnis mit `__interrupt__`-Schlüssel zurück
4. Mensch sieht den Interrupt-Wert und entscheidet
5. `graph.invoke(Command(resume=entscheidung), config=cfg)` – Graph setzt fort
6. `interrupt()` gibt den Resume-Wert zurück – Node verarbeitet ihn

**Drei HITL-Patterns im Überblick:**

| Pattern | Ablauf | Wann |
|---------|--------|------|
| **Approval** | Agent schlägt vor → Mensch genehmigt/lehnt ab | Irreversible Aktionen |
| **Edit** | Agent erstellt Entwurf → Mensch ändert → Weiter | Content-Erstellung |
| **Tool-Freigabe** | Agent plant Tool → Mensch bestätigt → Ausführen | Externe API-Aufrufe |

> **Wichtig:** `interrupt()` kann mehrfach im selben Graphen aufgerufen werden –  
> jedes Mal wird ein neuer Checkpoint gespeichert und der Mensch befragt.

In [3]:
#@markdown   <p><font size="4" color='green'>  Drei HITL-Patterns</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph APP["🟢 Approval-Pattern"]
        direction TB
        A1["Agent\nVorschlag"] --> A2["interrupt()"]
        A2 -->|ja| A3["✅ Ausführen"]
        A2 -->|nein| A4["❌ Abbruch"]
    end

    subgraph EDIT["🟡 Edit-Pattern"]
        direction TB
        E1["Agent\nEntwurf"] --> E2["interrupt()"]
        E2 -->|"Korrektur"| E3["✅ Geändert\nWeiter"]
    end

    subgraph TOOL["🟠 Tool-Freigabe"]
        direction TB
        T1["Agent"] --> T2["interrupt()"]
        T2 -->|genehmigt| T3["🔧 ToolNode"]
        T2 -->|abgelehnt| T4["❌ Ende"]
        T3 --> T1
    end
'''

mermaid(diagram, width=800)

# 3 | interrupt() und Approval-Pattern
---

Das **Approval-Pattern** ist das häufigste HITL-Muster:  
Ein Agent erstellt einen Vorschlag – der Mensch genehmigt oder lehnt ab.

> **In diesem Notebook ist die Hauptdemo ein Briefing-Review.**  
> Der Graph stoppt vor der Ausgabe einer Research-Antwort und zeigt Entwurf, Quellenanalyse und Verbesserungsvorschläge.

**Beispiel:** Meeting- & Research-Briefing-Review
- `entwurf_node` – erstellt einen belegpflichtigen Briefing-Entwurf
- `genehmigung_node` – `interrupt()` pausiert und zeigt Entwurf + Review-Hinweise
- `veroeffentlichen_node` – gibt das Briefing frei oder blockiert die Ausgabe

```python
def genehmigung_node(state):
    antwort = interrupt({
        "frage": "Research-Antwort freigeben?",
        "entwurf": state["entwurf"],
        "analyse": state["analyse"],
    })
    return {"genehmigt": str(antwort).lower() == "ja"}
```


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from IPython.display import Image as IPImage

from genai_lib.model_config import WORKER

# State fuer den Briefing-Review-Workflow.
# Der Name bleibt kompatibel mit den folgenden Edit-Pattern-Zellen.
class RedaktionsState(TypedDict):
    thema: str
    entwurf: str
    analyse: str
    vorschlaege: list[str]
    genehmigt: bool
    ergebnis: str

llm = init_chat_model(WORKER)

def entwurf_node(state: RedaktionsState) -> dict:
    """LLM erstellt einen knappen, belegpflichtigen Briefing-Entwurf."""
    prompt = [
        SystemMessage(content=(
            "Du bist ein Meeting- & Research-Briefing-Agent. "
            "Erstelle eine knappe Antwort mit Quellenhinweis oder markiere fehlende Evidenz."
        )),
        HumanMessage(content=f"Erstelle einen Review-Entwurf für: {state['thema']}"),
    ]
    response = llm.invoke(prompt)
    return {"entwurf": response.content}

def analyse_node(state: RedaktionsState) -> dict:
    """Prüft Quellenbindung, Korpusgrenze und Verständlichkeit."""
    analyse = "Prüfung: Quellenbindung sichtbar? Korpusgrenze markiert? Antwort verständlich?"
    vorschlaege = [
        "Mindestens einen Quellenhinweis oder 'Nicht im Korpus' nennen.",
        "Unsicherheit nicht glätten, sondern offen markieren.",
    ]
    return {"analyse": analyse, "vorschlaege": vorschlaege}

def genehmigung_node(state: RedaktionsState) -> dict:
    """Pausiert den Graphen und wartet auf menschliche Review-Entscheidung."""
    antwort = interrupt({
        "frage": "Research-Antwort freigeben? (ja/nein)",
        "entwurf": state["entwurf"],
        "analyse": state["analyse"],
        "vorschlaege": state["vorschlaege"],
    })
    return {"genehmigt": str(antwort).lower().strip() == "ja"}

def veroeffentlichen_node(state: RedaktionsState) -> dict:
    """Gibt das Briefing frei oder blockiert es."""
    if state["genehmigt"]:
        return {"ergebnis": f"Freigegebenes Briefing:\n{state['entwurf']}"}
    return {"ergebnis": "Nicht veröffentlicht: Review-Freigabe fehlt."}

# Graph aufbauen
builder_r = StateGraph(RedaktionsState)
builder_r.add_node("entwurf", entwurf_node)
builder_r.add_node("analyse", analyse_node)
builder_r.add_node("genehmigung", genehmigung_node)
builder_r.add_node("veroeffentlichen", veroeffentlichen_node)

builder_r.add_edge(START, "entwurf")
builder_r.add_edge("entwurf", "analyse")
builder_r.add_edge("analyse", "genehmigung")
builder_r.add_edge("genehmigung", "veroeffentlichen")
builder_r.add_edge("veroeffentlichen", END)

memory_r = InMemorySaver()
redaktions_graph = builder_r.compile(checkpointer=memory_r)
print("✅ Briefing-Review-Graph kompiliert")

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
display(IPImage(redaktions_graph.get_graph().draw_mermaid_png()))

In [ ]:
import uuid

run_cfg = {"run_name": "M17-Kap3-Approval-HITL", "tags": ["m17", "approval"]}
# Eindeutige thread_id verhindert Konflikte beim Mehrfach-Ausführen
cfg1 = {"configurable": {"thread_id": f"redaktion-{uuid.uuid4().hex[:6]}"}, **run_cfg}

# ── Schritt 1: LLM erstellt Entwurf ──────────────────────────────────────────
result = redaktions_graph.invoke(
    {"thema": "Warum braucht RAG-Evaluation sichtbare Quellenbindung?",
     "entwurf": "", "analyse": "", "vorschlaege": [], "genehmigt": False, "ergebnis": ""},
    config=cfg1
)

# ── Schritt 2: Review-Entscheidung ───────────────────────────────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸️ Graph pausiert – Review-Entscheidung erforderlich\n\n"
           f"**{val['frage']}**\n\n"
           f"**Entwurf:**\n> {val['entwurf']}\n\n**Analyse:** {val.get('analyse', '')}\n\n**Vorschläge:** {', '.join(val.get('vorschlaege', []))}")

    # ★ Zelle wartet hier auf Eingabe
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"
        print("  (Unbekannte Eingabe – als 'nein' gewertet)")

    # ── Schritt 3: Graph mit Entscheidung fortsetzen ─────────────────────────
    result = redaktions_graph.invoke(Command(resume=entscheidung), config=cfg1)
    mprint(f"\n**Ergebnis:** {result['ergebnis']}")

# 4 | HITL implementieren: Edit-Pattern (optional)
---

Dieses optionale Transfermuster bleibt im Briefing-Kontext: Das **Edit-Pattern** ermöglicht nicht nur Ja/Nein, sondern eine **inhaltliche Änderung**:

1. Agent erstellt Entwurf
2. Mensch sieht den Entwurf und schickt eine korrigierte Version
3. Graph arbeitet mit der korrigierten Version weiter

> **In diesem Notebook:** Ein eigener Text kann eingegeben werden; Enter übernimmt den Entwurf.  
> um den Originalentwurf zu übernehmen.

**Option A – Resume mit geändertem Wert (empfohlen):**

```python
def edit_node(state):
    korrektur = interrupt({"entwurf": state["entwurf"]})
    neuer_entwurf = str(korrektur).strip() or state["entwurf"]  # leer → original
    return {"entwurf": neuer_entwurf, "genehmigt": True}

# Aufruf-Seite
result   = graph.invoke(input_data, config=cfg)
entwurf  = result["__interrupt__"][0].value["entwurf"]
review_text = input("Text (Enter = übernehmen): ").strip()
result   = graph.invoke(Command(resume=review_text or entwurf), config=cfg)
```

**Option B – `update_state()` vor dem Resume:**

```python
graph.update_state(cfg, {"entwurf": "Korrigierter Text..."}, as_node="edit")
graph.invoke(Command(resume="ok"), config=cfg)
```

**Empfehlung:** Option A ist klarer und vollständig in LangSmith tracebar.

In [ ]:
# === Edit-Pattern: Entwurf verbessern ===

def edit_node(state: RedaktionsState) -> dict:
    """Entwurf an den Menschen senden; Korrektur kommt via Command(resume=...)."""
    korrektur = interrupt({
        "nachricht": "Bitte den Entwurf prüfen und ggf. ändern:",
        "entwurf":   state["entwurf"],
    })
    neuer_entwurf = str(korrektur).strip() or state["entwurf"]
    return {"entwurf": neuer_entwurf, "genehmigt": True}

builder_e = StateGraph(RedaktionsState)
builder_e.add_node("entwurf",         entwurf_node)
builder_e.add_node("edit",             edit_node)
builder_e.add_node("veroeffentlichen", veroeffentlichen_node)
builder_e.add_edge(START,      "entwurf")
builder_e.add_edge("entwurf",  "edit")
builder_e.add_edge("edit",     "veroeffentlichen")
builder_e.add_edge("veroeffentlichen", END)

memory_e   = InMemorySaver()
edit_graph = builder_e.compile(checkpointer=memory_e)

**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_edge(...)` — verbindet zwei Knoten mit einer festen Kante
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
display(IPImage(edit_graph.get_graph().draw_mermaid_png()))

In [ ]:
cfg_e = {"configurable": {"thread_id": f"edit-{uuid.uuid4().hex[:6]}"},
         "run_name": "M17-Kap4-Edit-HITL", "tags": ["m17", "edit"]}

# ── Schritt 1: LLM erstellt Entwurf ──────────────────────────────────────────
result = edit_graph.invoke(
    {"thema": "Review-Fassung für eine Antwort zu RAG-Evaluation",
     "entwurf": "", "analyse": "", "vorschlaege": [], "genehmigt": False, "ergebnis": ""},
    config=cfg_e
)

# ── Schritt 2: Review-Eingabe ────────────────────────────────────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸️ Entwurf zur Prüfung\n\n> {val['entwurf']}")

    print("\n★ Eigenen Text eintippen und Enter drücken")
    print("  (Nur Enter = Originalentwurf unverändert übernehmen)")
    eigener_text = input("\nIhr Text → ").strip()

    resume_text = eigener_text if eigener_text else val["entwurf"]
    if eigener_text:
        print("  → Der eingegebene Text wird verwendet.")
    else:
        print("  → Originalentwurf übernommen.")

    # ── Schritt 3: Fortsetzen ────────────────────────────────────────────────
    result = edit_graph.invoke(Command(resume=resume_text), config=cfg_e)
    mprint(f"\n**Ergebnis:** {result['ergebnis']}")

# 5 | Tool-Call-Genehmigung
---

Viele Agenten können **externe Aktionen** ausführen – E-Mails senden, Dateien ändern,  
APIs aufrufen. HITL-Freigabe vor dem Tool-Aufruf verhindert unerwünschte Effekte.

**Architektur:**
- `agent_node` – LLM plant Tool-Aufrufe
- `freigabe_node` – `interrupt()` zeigt geplante Tools, wartet auf Genehmigung
- `tools_node` – führt genehmigte Tools aus
- `abbruch_node` – bei Ablehnung: saubere Beendigung

**Alternativ: `interrupt_before` (deklarativ)**

```python
graph = builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]   # Graph pausiert automatisch VOR 'tools'
)
# Resume: graph.invoke(None, config=cfg)  – kein Command nötig
```

> **Empfehlung:** `interrupt()` im Node für komplexe Prüflogik;  
> `interrupt_before` für einfache Pause-vor-Ausführung-Szenarien.

In [22]:
#@markdown   <p><font size="4" color='green'>  Tool-Call-Genehmigung</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    START([START]) --> AGENT["🤖 Agent Node\n(llm.with_tools)"]

    AGENT --> COND{Tool Call?}

    COND -- "ja" --> FREIG[["⏸️ Freigabe-Node\ninterrupt()"]]
    COND -- "nein" --> END([END])

    FREIG -->|"genehmigt"| TOOLS["🔧 Tool Node"]
    FREIG -->|"abgelehnt"| FEEDBACK["💬 Feedback an Agent"]

    TOOLS --> AGENT
    FEEDBACK --> AGENT

    style FREIG   fill:#FF9800,stroke:#333,color:#fff
    style TOOLS   fill:#2196F3,stroke:#333,color:#fff
    style FEEDBACK fill:#F44336,stroke:#333,color:#fff
    style COND    fill:#fff,stroke:#333
'''

mermaid(diagram, width=700)

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode

from genai_lib.model_config import WORKER
@tool
def sende_email(empfaenger: str, betreff: str, inhalt: str) -> str:
    """Sendet eine E-Mail an den Emfaenger."""
    return f"E-Mail an {empfaenger} gesendet: '{betreff}'"

@tool
def erstelle_bericht(titel: str, inhalt: str) -> str:
    """Erstellt einen Bericht mit Titel und Inhalt."""
    return f"Bericht '{titel}' erstellt ({len(inhalt)} Zeichen)"

@tool
def loesche_datei(pfad: str) -> str:
    """Loescht eine Datei am angegebenen Pfad."""
    return f"Datei '{pfad}' geloescht"

hitl_tools    = [sende_email, erstelle_bericht, loesche_datei]
llm_mit_tools = init_chat_model(WORKER).bind_tools(hitl_tools)

# State
class FreigabeState(TypedDict):
    messages:  Annotated[list, add_messages]
    genehmigt: bool

def agent_node(state: FreigabeState) -> dict:
    """Ruft LLM mit gebundenen Tools auf."""
    response = llm_mit_tools.invoke(state["messages"])
    return {"messages": [response]}

def freigabe_node(state: FreigabeState) -> dict:
    """Zeigt geplante Tool-Aufrufe und wartet auf Genehmigung."""
    letzter_msg = state["messages"][-1]
    tool_namen  = [tc["name"] for tc in letzter_msg.tool_calls]
    antwort = interrupt({
        "nachricht":  f"Geplante Tool-Aufrufe: {', '.join(tool_namen)}",
        "anweisung":  "Genehmigen? (ja/nein)",
        "tool_calls": letzter_msg.tool_calls,
    })
    return {"genehmigt": str(antwort).lower().strip() == "ja"}

def freigabe_router(state: FreigabeState) -> str:
    return "tools" if state.get("genehmigt") else "abbruch"

def agent_router(state: FreigabeState) -> str:
    """Weiterleitung: Tool-Calls vorhanden → freigabe; sonst → END."""
    letzter = state["messages"][-1] if state["messages"] else None
    if letzter and hasattr(letzter, "tool_calls") and letzter.tool_calls:
        return "freigabe"
    return END

def abbruch_node(state: FreigabeState) -> dict:
    return {"messages": [AIMessage(content="Tool-Ausführung wurde abgelehnt.")]}

# Graph – explizite path_maps für korrekte Visualisierung
builder_f = StateGraph(FreigabeState)
builder_f.add_node("agent",    agent_node)
builder_f.add_node("freigabe", freigabe_node)
builder_f.add_node("tools",    ToolNode(hitl_tools))
builder_f.add_node("abbruch",  abbruch_node)

builder_f.add_edge(START, "agent")

# path_map explizit angeben → alle Kanten für draw_mermaid_png sichtbar
builder_f.add_conditional_edges(
    "agent",
    agent_router,
    {"freigabe": "freigabe", END: END}
)
builder_f.add_conditional_edges(
    "freigabe",
    freigabe_router,
    {"tools": "tools", "abbruch": "abbruch"}
)
builder_f.add_edge("tools",   "agent")
builder_f.add_edge("abbruch", END)

memory_f       = InMemorySaver()
freigabe_graph = builder_f.compile(checkpointer=memory_f)


**Was passiert hier?**

1. `StateGraph(...)` — definiert den Graphen mit dem State-Schema
2. `add_node(...)` — registriert einen Knoten im Graphen
3. `add_conditional_edges(...)` — legt bedingte Übergänge zwischen Knoten fest
4. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
5. `compile(...)` — schließt den Graphen ab und erzeugt das ausführbare Objekt

In [ ]:
display(IPImage(freigabe_graph.get_graph().draw_mermaid_png()))

In [ ]:
run_cfg_f = {"run_name": "M17-Kap5-Genehmigung-HITL", "tags": ["m17", "tool-approval"]}
cfg_f = {"configurable": {"thread_id": f"freigabe-{uuid.uuid4().hex[:6]}"}, **run_cfg_f}

# ── Schritt 1: Anfrage stellen – LLM plant Tool-Aufruf ───────────────────────
result = freigabe_graph.invoke(
    {"messages": [HumanMessage(content="Sende eine Zusammenfassung an chef@firma.de")],
     "genehmigt": False},
    config=cfg_f
)

# ── Schritt 2: Geplante Tools anzeigen + Entscheidung ───────────────────────
if "__interrupt__" in result:
    val = result["__interrupt__"][0].value

    # Geplante Aufrufe detailliert ausgeben
    tool_details = "\n".join(
        f"  • **{tc['name']}**"
        + (f"({', '.join(f'`{k}`={v!r}' for k, v in tc.get('args', {}).items())})"
           if tc.get("args") else "()")
        for tc in val.get("tool_calls", [])
    )
    mprint(f"## ⏸️ Tool-Freigabe erforderlich\n\n"
           f"**{val['nachricht']}**\n\n"
           f"Geplante Aufrufe:\n{tool_details}\n\n"
           f"*{val['anweisung']}*")

    # ★ Zelle wartet auf Eingabe
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"
        print("  (Unbekannte Eingabe – als 'nein' gewertet)")

    # ── Schritt 3: Graph fortsetzen ──────────────────────────────────────────
    result = freigabe_graph.invoke(Command(resume=entscheidung), config=cfg_f)
    mprint(f"\n**Ergebnis:** {result['messages'][-1].content}")

# 6 | LangSmith: HITL-Traces
---

HITL-Workflows erzeugen in LangSmith besondere Trace-Strukturen:  
Ein unterbrochener Graph erscheint als **zweistufiger Run** –  
invoke_1 (bis interrupt) + invoke_2 (nach resume).

**Was in LangSmith sichtbar ist:**

| Element | Beschreibung |
|---------|-------------|
| **Interrupted Run** | Run-Status `interrupted` bis zum Resume |
| **Resume Run** | Neuer Run mit gleichem `thread_id`, angedockt an den ersten |
| **Interrupt-Wert** | In den Metadaten des Node-Spans sichtbar |
| **user Decision** | Resume-Wert erscheint als Input des zweiten Runs |

**Best Practices für HITL-Tracing:**

```python
cfg = {
    "configurable": {"thread_id": "hitl-session-42"},
    "run_name":  "HITL-Content-Freigabe",
    "tags":      ["m17", "hitl", "approval", "marketing"],
    "metadata":  {
        "user_id":    "user-42",
        "workflow":   "content-approval",
        "version":    "1.0",
    }
}
```

> **Tipp:** Verwende aussagekräftige `run_name`-Werte für HITL-Runs,  
> z.B. `"Freigabe-Entwurf"` statt `"invoke"`. Das erleichtert das Debugging  
> erheblich, wenn viele parallele HITL-Sessions aktiv sind.  
> *LangSmith Evaluations Basics* zeigt die vollständige Trace-Analyse.

In [ ]:
# LangSmith-optimiertes HITL-Beispiel
# ──────────────────────────────────────────────────────────────────────────────
# In LangSmith beobachten:
#   Run 1 → Status "interrupted" (bis zur Eingabe)
#   Run 2 → Status "success"    (nach Command(resume=...))
# ──────────────────────────────────────────────────────────────────────────────
cfg_ls = {
    "configurable": {"thread_id": f"hitl-ls-{uuid.uuid4().hex[:6]}"},
    "run_name": "M17-Kap6-Traces",
    "tags":     ["m17", "hitl", "langsmith-demo"],
    "metadata": {
        "user_id":  "trainer-42",
        "workflow": "content-approval",
    }
}

result = redaktions_graph.invoke(
    {"thema": "LangSmith-Tracing", "entwurf": "", "analyse": "", "vorschlaege": [], "genehmigt": False, "ergebnis": ""},
    config=cfg_ls
)

if "__interrupt__" in result:
    val = result["__interrupt__"][0].value
    mprint(f"## ⏸️ LangSmith sieht diesen Run als 'interrupted'\n\n"
           f"**Entwurf:**\n> {val['entwurf']}\n\n**Analyse:** {val.get('analyse', '')}\n\n**Vorschläge:** {', '.join(val.get('vorschlaege', []))}")

    # ★ Echte Eingabe – LangSmith protokolliert die Entscheidung als Resume-Input
    entscheidung = input("\nIhre Entscheidung (ja / nein) → ").strip().lower()
    if entscheidung not in ("ja", "nein"):
        entscheidung = "nein"

result = redaktions_graph.invoke(Command(resume=entscheidung), config=cfg_ls)
print(f"\nErgebnis: {result['ergebnis'][:100]}")
print("\nIn LangSmith sichtbar:")
print(f"  Run 1  → 'HITL-Marketing-Freigabe'  Status: interrupted")
print(f"  Run 2  → Resume-Run                 Status: success")
print(f"  Input  → Command(resume='{entscheidung}')")
print(f"  Tags   → ['m17', 'hitl', 'langsmith-demo']")

In [15]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M17-Human-in-the-Loop", limit=3, show_steps=True)

## LangSmith Trace — `M17-Human-in-the-Loop`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M17-Kap6-Traces` | ✅ success | 0.0s | 0 |
| `M17-Kap6-Traces` | ✅ success | 2.4s | 0 |
| `M17-Kap5-Genehmigung-HITL` | ✅ success | 0.9s | 0 |


### Steps — letzter Run: `M17-Kap6-Traces`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `veroeffentlichen` | ✅ | 0.0s |
| 2 | `chain` | `genehmigung` | ✅ | 0.0s |

# A | Aufgaben|# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen|Die Aufgabenstellungen bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> Generative KI kann als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei Fehlermeldungen, Teilproblemen oder Code-Varianten kann zum Beispiel Gemini in Google Colab helfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Human-in-the-Loop für den Meeting- & Research-Briefing-Agent
</font></p>

Der Meeting- & Research-Briefing-Agent soll Vorschläge nicht blind übernehmen. Kritische Änderungen, etwa eine neue Antwortfassung oder eine Quellenentscheidung, werden vor der Anwendung durch einen menschlichen Review-Schritt freigegeben.


**Grundlagen**
- Einen `ResearchReviewState` mit Analyse, Vorschlägen, Genehmigung und finalem Text definieren.
- Einen Graphen mit `interrupt()` im Review-Node bauen.
- Den kompilierten Graphen in `mein_graph` speichern.

**✅ Erledigt wenn:** `mein_graph.invoke(...)` stoppt im Review-Node und wartet auf Freigabe.


In [16]:
# Grundlagen: Review-Graph mit interrupt()
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ResearchReviewState(TypedDict):
    frage: str
    entwurf: str
    analyse: str
    vorschlaege: list[str]
    genehmigt: bool
    finaler_text: str
    manuelle_korrektur: str
    iter_count: int
    max_iter: int


def analyse_node(state: ResearchReviewState) -> dict:
    analyse = "Der Entwurf wird auf Quellenbindung, Korpusgrenze und Verständlichkeit geprüft."
    return {"analyse": analyse}


def vorschlag_node(state: ResearchReviewState) -> dict:
    vorschlaege = [
        "Quellenbindung sichtbar machen.",
        "Bei fehlender Evidenz klar 'Nicht im Korpus' schreiben.",
    ]
    return {"vorschlaege": vorschlaege}


def review_node(state: ResearchReviewState) -> dict:
    entscheidung = interrupt({
        "frage": "Research-Antwort freigeben?",
        "entwurf": state["entwurf"],
        "analyse": state["analyse"],
        "vorschlaege": state["vorschlaege"],
    })
    if isinstance(entscheidung, dict):
        genehmigt = bool(entscheidung.get("genehmigt", False))
        manuelle_korrektur = str(entscheidung.get("manuelle_korrektur", ""))
    else:
        text = str(entscheidung).strip().lower()
        genehmigt = text in {"ja", "j", "true", "freigeben", "genehmigt"}
        manuelle_korrektur = ""
    return {"genehmigt": genehmigt, "manuelle_korrektur": manuelle_korrektur}


def anwenden_node(state: ResearchReviewState) -> dict:
    if state.get("manuelle_korrektur"):
        finaler_text = state["manuelle_korrektur"]
    else:
        finaler_text = state["entwurf"] + "\n\nFreigegeben nach menschlichem Review."
    return {"finaler_text": finaler_text}


def ablehnen_node(state: ResearchReviewState) -> dict:
    finaler_text = "Nicht veröffentlicht: Review-Freigabe fehlt."
    return {"finaler_text": finaler_text}


def route_review(state: ResearchReviewState) -> str:
    return "anwenden" if state.get("genehmigt") else "ablehnen"


builder = StateGraph(ResearchReviewState)
builder.add_node("analyse", analyse_node)
builder.add_node("vorschlag", vorschlag_node)
builder.add_node("review", review_node)
builder.add_node("anwenden", anwenden_node)
builder.add_node("ablehnen", ablehnen_node)

builder.add_edge(START, "analyse")
builder.add_edge("analyse", "vorschlag")
builder.add_edge("vorschlag", "review")
builder.add_conditional_edges("review", route_review, {"anwenden": "anwenden", "ablehnen": "ablehnen"})
builder.add_edge("anwenden", END)
builder.add_edge("ablehnen", END)

memory = InMemorySaver()
mein_graph = builder.compile(checkpointer=memory)
print("✅ Research-Review-Graph kompiliert")


✅ Research-Review-Graph kompiliert


In [17]:
# ✅ Selbstcheck Grundlagen
_cfg_grundlagen = {"configurable": {"thread_id": "m17-selfcheck-grundlagen"}}
_start_state = {
    "frage": "Warum verbessert RAG die Zuverlässigkeit?",
    "entwurf": "RAG verbessert Antworten durch zusätzliche Quellen.",
    "analyse": "",
    "vorschlaege": [],
    "genehmigt": False,
    "finaler_text": "",
    "manuelle_korrektur": "",
    "iter_count": 0,
    "max_iter": 3,
}

assert hasattr(mein_graph, "invoke"), "❌ mein_graph muss invoke() unterstützen."
mein_graph.invoke(_start_state, config=_cfg_grundlagen)
_state = mein_graph.get_state(_cfg_grundlagen)
assert len(_state.next) > 0, "❌ Der Graph hat nicht beim interrupt() pausiert."
assert "review" in _state.next, "❌ Der nächste Schritt muss der Review-Node sein."

print("✅ Grundlagen-Selbstcheck bestanden!")


✅ Grundlagen-Selbstcheck bestanden!


**Aufbau**
- Einen Approve-Fall und einen Reject-Fall mit `Command(resume=...)` ausführen.
- Die finalen Texte vergleichen.
- Ergebnisse in `result_approve` und `result_reject` speichern.

**✅ Erledigt wenn:** Der Approve-Fall veröffentlicht Text, der Reject-Fall blockiert die Veröffentlichung.


In [18]:
# Aufbau: Approve und Reject durchspielen
approve_cfg = {"configurable": {"thread_id": "m17-approve"}}
reject_cfg = {"configurable": {"thread_id": "m17-reject"}}

basis_state = {
    "frage": "Wie unterscheidet sich naive RAG von advanced RAG?",
    "entwurf": "Naive RAG ruft einfache Kontexte ab; advanced RAG ergänzt Optimierungen im Retrieval.",
    "analyse": "",
    "vorschlaege": [],
    "genehmigt": False,
    "finaler_text": "",
    "manuelle_korrektur": "",
    "iter_count": 0,
    "max_iter": 3,
}

mein_graph.invoke(basis_state, config=approve_cfg)
result_approve = mein_graph.invoke(Command(resume="ja"), config=approve_cfg)

mein_graph.invoke(basis_state, config=reject_cfg)
result_reject = mein_graph.invoke(Command(resume="nein"), config=reject_cfg)

print("Approve:", result_approve["finaler_text"])
print("Reject:", result_reject["finaler_text"])


Approve: Naive RAG ruft einfache Kontexte ab; advanced RAG ergänzt Optimierungen im Retrieval.

Freigegeben nach menschlichem Review.
Reject: Nicht veröffentlicht: Review-Freigabe fehlt.


In [19]:
# ✅ Selbstcheck Aufbau
assert result_approve.get("genehmigt") is True, "❌ Approve-Fall wurde nicht genehmigt."
assert "Freigegeben" in result_approve.get("finaler_text", ""), (
    "❌ Approve-Fall muss einen freigegebenen finalen Text liefern."
)
assert result_reject.get("genehmigt") is False, "❌ Reject-Fall darf nicht genehmigt sein."
assert "Nicht veröffentlicht" in result_reject.get("finaler_text", ""), (
    "❌ Reject-Fall muss die Veröffentlichung blockieren."
)

print("✅ Aufbau-Selbstcheck bestanden!")


✅ Aufbau-Selbstcheck bestanden!


**Vertiefung**
- Eine manuelle Korrektur per `Command(resume={...})` übergeben.
- Die Korrektur als finalen Text übernehmen.
- Eine einfache Iterationsgrenze im State dokumentieren.

**✅ Erledigt wenn:** Die manuelle Korrektur erscheint unverändert in `result_manuell["finaler_text"]`.


In [20]:
# Vertiefung: manuelle Korrektur übernehmen
manual_cfg = {"configurable": {"thread_id": "m17-manuell"}}
manual_state = {
    "frage": "Warum ist Evaluation bei RAG wichtig?",
    "entwurf": "Evaluation prüft, ob Antworten nützlich sind.",
    "analyse": "",
    "vorschlaege": [],
    "genehmigt": False,
    "finaler_text": "",
    "manuelle_korrektur": "",
    "iter_count": 1,
    "max_iter": 3,
}

manuelle_fassung = (
    "Evaluation bei RAG prüft getrennt Retrieval-Qualität, Quellenbindung und Antwortqualität."
)
mein_graph.invoke(manual_state, config=manual_cfg)
result_manuell = mein_graph.invoke(
    Command(resume={"genehmigt": True, "manuelle_korrektur": manuelle_fassung}),
    config=manual_cfg,
)

iterationsgrenze_ok = manual_state["iter_count"] < manual_state["max_iter"]

print(result_manuell["finaler_text"])
print("Iterationsgrenze offen:", iterationsgrenze_ok)


Evaluation bei RAG prüft getrennt Retrieval-Qualität, Quellenbindung und Antwortqualität.
Iterationsgrenze offen: True


In [21]:
# ✅ Selbstcheck Vertiefung
assert result_manuell.get("genehmigt") is True, "❌ Manuelle Fassung wurde nicht genehmigt."
assert result_manuell.get("finaler_text") == manuelle_fassung, (
    "❌ Die manuelle Korrektur muss unverändert übernommen werden."
)
assert iterationsgrenze_ok is True, "❌ Iterationsgrenze ist nicht plausibel gesetzt."

print("✅ Vertiefung-Selbstcheck bestanden!")


✅ Vertiefung-Selbstcheck bestanden!
